In [5]:
# import pandas as pd

# # 初始化一个空字典，用于存储所有站点经纬度信息
# all_site_latlon_dict = {}

# # 遍历2001到2019年
# for year in range(2001, 2020):
#     # 构建文件路径
#     file_path = f'/backupdata/data_EPA/EQUATES/EQUATES_data/ds.input.aqs.o3.{year}.csv'
#     try:
#         # 读取CSV文件
#         df = pd.read_csv(file_path)
#         # 提取唯一站点及其经纬度
#         unique_sites = df.drop_duplicates(subset='Site')[['Site', 'Lat', 'Lon']]
#         # 将站点ID转换为字符串类型
#         unique_sites['Site'] = unique_sites['Site'].astype(str)
#         # 遍历每个唯一站点，更新总字典
#         for _, row in unique_sites.iterrows():
#             site = row['Site']
#             lat = row['Lat']
#             lon = row['Lon']
#             all_site_latlon_dict[site] = {'Lat': lat, 'Lon': lon}
#     except FileNotFoundError:
#         print(f"文件 {file_path} 未找到。")
#     except Exception as e:
#         print(f"读取文件 {file_path} 时出现错误: {e}")

# # 处理新加入的文件
# new_files = [
#     '/backupdata/data_EPA/EQUATES/EQUATES_data/SMAT_OZONE_MDA1_APRSEP_STD70_2000_2022.CSV',
#     '/backupdata/data_EPA/EQUATES/EQUATES_data/SMAT_OZONE_MDA8_MAYSEP_STD70_2000_2022.CSV'
# ]

# for file_path in new_files:
#     try:
#         df = pd.read_csv(file_path)
#         # 假设新文件里的站点和经纬度列名和之前的一样，如果不同需修改
#         unique_sites = df.drop_duplicates(subset='Site')[['Site', 'Lat', 'Lon']]
#         unique_sites['Site'] = unique_sites['Site'].astype(str)
#         for _, row in unique_sites.iterrows():
#             site = row['Site']
#             lat = row['Lat']
#             lon = row['Lon']
#             all_site_latlon_dict[site] = {'Lat': lat, 'Lon': lon}
#     except FileNotFoundError:
#         print(f"文件 {file_path} 未找到。")
#     except Exception as e:
#         print(f"读取文件 {file_path} 时出现错误: {e}")

# # 处理 MonitorsTimeRegion_Filter.csv 文件
# monitors_file = '/output/Region/MonitorsTimeRegion_Filter.csv'
# try:
#     df = pd.read_csv(monitors_file)
#     if 'site_id' in df.columns and 'Lat' in df.columns and 'Lon' in df.columns:
#         unique_sites = df.drop_duplicates(subset='site_id')[['site_id', 'Lat', 'Lon']]
#         unique_sites['site_id'] = unique_sites['site_id'].astype(str)
#         for _, row in unique_sites.iterrows():
#             site = row['site_id']
#             lat = row['Lat']
#             lon = row['Lon']
#             all_site_latlon_dict[site] = {'Lat': lat, 'Lon': lon}
#     else:
#         print(f"文件 {monitors_file} 中缺少必要的列（site_id, Lat, Lon）。")
# except FileNotFoundError:
#     print(f"文件 {monitors_file} 未找到。")
# except Exception as e:
#     print(f"读取文件 {monitors_file} 时出现错误: {e}")

# # 将汇总的字典转换为DataFrame
# result_df = pd.DataFrame.from_dict(all_site_latlon_dict, orient='index')
# result_df.index.name = 'Site'

# # 保存结果到CSV文件
# output_path = '/backupdata/data_EPA/EQUATES/EQUATES_data/SitesTable2001-2020.csv'
# try:
#     result_df.to_csv(output_path)
#     print(f"结果已成功保存到 {output_path}")
# except Exception as e:
#     print(f"保存文件时出现错误: {e}")

# # 输出结果数据表中唯一站点的信息
# unique_sites = result_df.index.unique()
# print("输出数据表中的唯一站点:")
# print(unique_sites)

In [1]:
import pandas as pd
import re

# 读取第一个文件，提取 UniqueSite 和对应的 LatLon
file_path_1 = '/backupdata/data_EPA/EQUATES/EQUATES_data/SitesTable2001-2020.csv'
df_1 = pd.read_csv(file_path_1)
unique_sites = df_1.drop_duplicates(subset='Site')[['Site', 'Lat', 'Lon']]
unique_sites['Site'] = unique_sites['Site'].astype(str)
# 过滤掉带字母的站点
valid_sites = unique_sites[~unique_sites['Site'].str.contains(r'[a-zA-Z]', regex=True)]
site_latlon_dict = valid_sites.set_index('Site')[['Lat', 'Lon']].to_dict(orient='index')
# 找出被过滤掉的站点
filtered_out_sites_1 = set(unique_sites['Site']) - set(valid_sites['Site'])
print(f"从站点表中过滤掉的站点: {filtered_out_sites_1}")

# 站点表
unique_site_count = valid_sites['Site'].nunique()

# 要处理的年份列表
years = [2013]

for year in years:
    # 读取第二个文件，只读取需要的列
    file_path_2 = f'/backupdata/data_EPA/aq_obs/routine/{year}/AQS_hourly_data_{year}.csv'
    try:
        df_2 = pd.read_csv(file_path_2, usecols=['site_id', 'POCode', 'dateon', 'O3'])
    except Exception as e:
        print(f"读取 {year} 年文件时出现错误: {e}")
        continue

    filtered_df = df_2[(df_2['O3'] != -999)]
    filtered_df['site_id'] = filtered_df['site_id'].astype(str)
    # 过滤掉带字母的站点
    valid_input_sites = filtered_df[~filtered_df['site_id'].str.contains(r'[a-zA-Z]', regex=True)]
    # 找出被过滤掉的站点
    filtered_out_sites_2 = set(filtered_df['site_id']) - set(valid_input_sites['site_id'])
    print(f"{year} 年从输入文件中过滤掉的站点: {filtered_out_sites_2}")
    # 统计输入文件中的唯一站点个数
    input_unique_site_count = valid_input_sites['site_id'].nunique()

    # 添加经纬度信息
    valid_input_sites['Lat'] = valid_input_sites['site_id'].map(lambda x: site_latlon_dict.get(x, {}).get('Lat'))
    valid_input_sites['Lon'] = valid_input_sites['site_id'].map(lambda x: site_latlon_dict.get(x, {}).get('Lon'))

    # 剔除不能添加上经纬度的行
    valid_input_sites = valid_input_sites.dropna(subset=['Lat', 'Lon'])

    # 统计输出文件中的唯一站点个数
    output_unique_site_count = valid_input_sites['site_id'].nunique()

    # 输出结果到指定文件
    output_path = f'/backupdata/data_EPA/aq_obs/routine/{year}/AQS_hourly_data_{year}_LatLon.csv'
    valid_input_sites.to_csv(output_path, index=False)

    print(f"{year} 年站点表中的唯一站点个数: {unique_site_count}")
    print(f"{year} 年输入文件中O3站点的数据行数: {input_unique_site_count}")
    print(f"{year} 年输出文件中O3站点个数: {output_unique_site_count}")
    print(f"{year} 年结果已成功保存到 {output_path}")    

从站点表中过滤掉的站点: set()


/tmp/ipykernel_81994/2130758419.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['site_id'] = filtered_df['site_id'].astype(str)


2013 年从输入文件中过滤掉的站点: set()
2013 年站点表中的唯一站点个数: 2016
2013 年输入文件中O3站点的数据行数: 1316
2013 年输出文件中O3站点个数: 1314
2013 年结果已成功保存到 /backupdata/data_EPA/aq_obs/routine/2013/AQS_hourly_data_2013_LatLon.csv


In [2]:
# import pandas as pd

# # 定义输入和输出文件路径
# input_file = '/DeepLearning/mnt/shixiansheng/data_fusion/output/W126/2011_W126_ST.csv'
# output_file = '/DeepLearning/mnt/shixiansheng/data_fusion/output/W126/2011_Model_W126_ST.csv'

# try:
#     # 读取 CSV 文件
#     df = pd.read_csv(input_file)

#     # 提取所需的列
#     selected_columns = df[['ROW', 'COL', 'model', 'Period']]

#     # 将提取的列保存为新的 CSV 文件
#     selected_columns.to_csv(output_file, index=False)
#     print(f"已成功提取指定列并保存到 {output_file}")
# except FileNotFoundError:
#     print(f"错误: 未找到文件 {input_file}")
# except Exception as e:
#     print(f"发生未知错误: {e}")
    

In [3]:
# import pandas as pd

# # 读取 CSV 文件
# file_path = '/DeepLearning/mnt/shixiansheng/data_fusion/output/W126/2011_W126_ST_AtF_True.csv'
# df = pd.read_csv(file_path)

# # 添加新列
# df['Period'] = 'W126'

# # 保存修改后的数据到原文件
# df.to_csv(file_path, index=False)

# print("已成功添加 'Period' 列到 CSV 文件。")

In [4]:
# import pandas as pd

# file_path = '/DeepLearning/mnt/shixiansheng/data_fusion/output/W126/2004_Monitor_W126.csv'
# df = pd.read_csv(file_path)

# count_site_id = df['site_id'].count()
# count_O3 = df['O3'].count()
# count_Period = df['Period'].count()
# print(f"site_id的非空值数量: {count_site_id}")
# print(f"O3的非空值数量: {count_O3}")
# print(f"Period的非空值数量: {count_Period}")

# mean_O3 = df['O3'].mean()
# print(f"O3的均值: {mean_O3}")

# std_O3 = df['O3'].std()
# print(f"O3的标准差: {std_O3}")

# min_O3 = df['O3'].min()
# print(f"O3的最小值: {min_O3}")

# q25_O3 = df['O3'].quantile(0.25)
# print(f"O3的25%分位数: {q25_O3}")

# median_O3 = df['O3'].median()
# print(f"O3的中位数: {median_O3}")

# q75_O3 = df['O3'].quantile(0.75)
# print(f"O3的75%分位数: {q75_O3}")

# max_O3 = df['O3'].max()
# print(f"O3的最大值: {max_O3}")